# P2 — Vietnamese News Classification
## Phase 2 — Vietnamese NLP + TF-IDF Preprocessing

**Dataset**: UVN-1 (3,246 samples, 13 classes)

**Mục tiêu Phase 2**:
- Xây dựng feature pipeline: cleaning → underthesea → stopwords → TF-IDF
- Chạy ablation E0 → E3
- Lưu tokenized dataset + metadata

**KHÔNG làm ở Phase 2**:
- ❌ Train model chính thức (sang Phase 3)
- ❌ Benchmark NB/LR/SVM (sang Phase 3)
- ❌ Đánh giá trên test set (sang Phase 4)

**Data Leakage Protocol**:
- TRAIN: fit TF-IDF
- VALIDATION: transform only
- TEST: 🔒 LOCKED — KHÔNG load trong Phase 2

---
# Section 0 — Setup & Config

In [ ]:
# ============================================================
# 0.1 — Imports
# ============================================================
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

from underthesea import word_tokenize

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("✅ Imports OK")

In [ ]:
# ============================================================
# 0.2 — Config: paths, version, seed
# ============================================================
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
STOPWORDS_PATH = PROJECT_ROOT / "data" / "stopwords_vi.txt"
CACHE_PATH = DATA_INTERIM / "news_tokenized.csv"

# Versions
DATASET_VERSION = "uvn1-v1.0.0"
STOPWORDS_VERSION = "vi-v1.0.0"
UNDERTHESEA_VERSION_EXPECTED = "9.5.0"
RANDOM_SEED = 42

# Verify files exist
required = [
    DATA_PROCESSED / "train.csv",
    DATA_PROCESSED / "validation.csv",
    DATA_PROCESSED / "categories.json",
    STOPWORDS_PATH,
]
for p in required:
    status = "✅" if p.exists() else "❌"
    print(f"{status} {p}")

print(f"\nPROJECT_ROOT      : {PROJECT_ROOT}")
print(f"DATASET_VERSION   : {DATASET_VERSION}")
print(f"STOPWORDS_VERSION : {STOPWORDS_VERSION}")

In [ ]:
# ============================================================
# 0.3 — Verify underthesea version (FAIL-FAST)
# ============================================================
import underthesea
actual_ver = underthesea.__version__
print(f"underthesea version: {actual_ver}")

if actual_ver != UNDERTHESEA_VERSION_EXPECTED:
    raise RuntimeError(
        f"Expected underthesea {UNDERTHESEA_VERSION_EXPECTED}, "
        f"but found {actual_ver}. "
        f"Vui lòng cài đúng version để tránh cache inconsistency."
    )

print("✅ Version matches expected")

# Test word_tokenize
sample = "ABBANK được chấp thuận chào bán cổ phiếu để nâng vốn điều lệ"
tokens = word_tokenize(sample)
print(f"\nSample raw : {sample}")
print(f"Sample tok : {tokens}")

---
# Section 1 — Load Data (02.1)

In [ ]:
# ============================================================
# 1.1 — Load train/val ONLY (test LOCKED)
# ============================================================
train_df = pd.read_csv(DATA_PROCESSED / "train.csv", encoding="utf-8-sig")
val_df = pd.read_csv(DATA_PROCESSED / "validation.csv", encoding="utf-8-sig")

print(f"Train      : {train_df.shape}")
print(f"Validation : {val_df.shape}")
print("Test       : 🔒 LOCKED — not loaded in Phase 2")
print(f"\nColumns: {list(train_df.columns)}")

In [ ]:
# ============================================================
# 1.2 — Load categories.json
# ============================================================
with open(DATA_PROCESSED / "categories.json", encoding="utf-8") as f:
    categories_meta = json.load(f)

CATEGORIES = categories_meta["categories"]
print(f"Number of categories: {len(CATEGORIES)}")
print(f"Categories: {CATEGORIES}")

In [ ]:
# ============================================================
# 1.3 — Class distribution (train + val only)
# ============================================================
dist = pd.DataFrame({
    "train": train_df["label"].value_counts(),
    "validation": val_df["label"].value_counts(),
}).fillna(0).astype(int)

dist["train_pct"] = (dist["train"] / dist["train"].sum() * 100).round(2)
dist = dist.sort_values("train", ascending=False)
display(dist)

print(f"\nMax/Min ratio (train): {dist['train'].max() / dist['train'].min():.1f}x")

**Nhận xét**:
- Imbalance nghiêm trọng giữa `Kinh doanh` và `Xe`
- Macro-F1 sẽ là metric chính
- `class_weight="balanced"` sẽ được test ở Phase 3

**Test set**: 🔒 LOCKED — không load, không xem phân bố trong Phase 2.

---
# Section 2 — Build text (02.2)

In [ ]:
# ============================================================
# 2.1 — Load stopwords
# ============================================================
def load_stopwords(path: Path) -> tuple[set, int]:
    """Load stopwords từ file, bỏ comment và dòng trống, dedup."""
    stopwords = set()
    raw_count = 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            raw_count += 1
            stopwords.add(line.lower())
    return stopwords, raw_count

STOPWORDS, RAW_COUNT = load_stopwords(STOPWORDS_PATH)

print(f"Raw candidates  : {RAW_COUNT}")
print(f"After dedup     : {len(STOPWORDS)}")
print(f"Duplicates      : {RAW_COUNT - len(STOPWORDS)}")

# Verify 13 category keywords NOT in stopwords
CATEGORY_KEYWORDS = {
    "công đoàn", "giáo dục", "giải trí", "khoa học", "kinh doanh",
    "pháp luật", "sức khỏe", "thế giới", "thể thao", "thời sự",
    "xe", "xã hội", "đời sống",
}
violations = CATEGORY_KEYWORDS & STOPWORDS
if violations:
    raise ValueError(f"CONTRACT VIOLATION: category keywords in stopwords: {violations}")
print(f"✅ Contract OK: none of 13 category keywords in stopwords")

# Verify news signal words NOT in stopwords
NEWS_SIGNAL = {"tin", "thông tin", "cho biết", "đưa tin", "đưa", "nói"}
violations2 = NEWS_SIGNAL & STOPWORDS
if violations2:
    print(f"⚠️  News signal words in stopwords: {violations2}")
else:
    print(f"✅ News signal words NOT in stopwords")

In [ ]:
# ============================================================
# 2.2 — Verify text == title + content (FULL DATASET)
# ============================================================
def verify_text_column(df: pd.DataFrame, name: str) -> None:
    """Verify toàn bộ dataset: text == title + ' ' + content."""
    expected = (
        df["title"].fillna("").astype(str)
        + " "
        + df["content"].fillna("").astype(str)
    ).str.strip()

    actual = df["text"].fillna("").astype(str).str.strip()

    mismatches = (expected != actual).sum()
    print(f"{name}: {mismatches} mismatches / {len(df)} rows")

    if mismatches:
        raise ValueError(f"{name}: text column verification failed")

    print(f"✅ {name}: text == title + content")


verify_text_column(train_df, "Train")
verify_text_column(val_df, "Validation")

In [ ]:
# ============================================================
# 2.3 — Text statistics (KHÔNG mutate DataFrame)
# ============================================================
for name, df in [("Train", train_df), ("Validation", val_df)]:
    text_len = df["text"].str.len()
    text_words = df["text"].str.split().str.len()

    print(f"\n{name}:")
    print(f"  Length (chars) — mean: {text_len.mean():.0f}, "
          f"median: {text_len.median():.0f}, "
          f"min: {text_len.min()}, "
          f"max: {text_len.max()}")
    print(f"  Words          — mean: {text_words.mean():.0f}, "
          f"median: {text_words.median():.0f}, "
          f"min: {text_words.min()}, "
          f"max: {text_words.max()}")

In [ ]:
# ============================================================
# 2.4 — Sample texts
# ============================================================
print("=" * 80)
print("SAMPLE 1 (first train sample)")
print("=" * 80)
print(f"Label  : {train_df.iloc[0]['label']}")
print(f"Title  : {train_df.iloc[0]['title']}")
print(f"Content (first 300 chars):")
print(train_df.iloc[0]['content'][:300] + "...")

print("\n" + "=" * 80)
print("SAMPLE 2 (index 100)")
print("=" * 80)
print(f"Label  : {train_df.iloc[100]['label']}")
print(f"Title  : {train_df.iloc[100]['title']}")
print(f"Content (first 300 chars):")
print(train_df.iloc[100]['content'][:300] + "...")

In [ ]:
# ============================================================
# 2.5 — Section 0-2 summary
# ============================================================
print("=" * 60)
print("SECTION 0-2 SUMMARY")
print("=" * 60)
print(f"✅ Setup OK")
print(f"✅ Loaded: train={len(train_df)}, val={len(val_df)}")
print(f"🔒 Test  : LOCKED (not loaded)")
print(f"✅ Categories: {len(CATEGORIES)}")
print(f"✅ Stopwords: {len(STOPWORDS)} unique ({RAW_COUNT} raw)")
print(f"✅ text column verified FULL dataset: title + content")
print(f"\n⏭️  NEXT: Section 3 — Baseline E0")